# Preprocessing: Allegheny County Jail Oversight Board minutes

This notebook includes code for scraping official meeting files, turning them into plain text, then normalizing and lemmatizing that text for downstream models.

## Section

1. **File download and text extraction** — from scraped PDF/DOCX to `Data/Text/*.txt` and `Data/Combined/raw_documents.pkl`.
2. **Cleaning and lemmatization** — HTML artifacts, punctuation, stopwords, and `spaCy` lemmatization; writes `Data/Text/lemmatized_text.xz`.

## Outputs used later in the project

| Artifact | Where it is written | Typical downstream use |
|----------|---------------------|-------------------------|
| `Data/Combined/raw_documents.pkl` | End of Section 1 | BERTopic (raw text plus year/month from filenames) |
| `Data/Text/*.txt` | Chapter 1 | Human inspection, optional reloads |
| `Data/Text/lemmatized_text.xz` | End of Section 2 | LDA, n-grams |

## Contributors

| Section | Main contributor |
|---------|------------------|
| 1. File download, text extraction (output: `raw_documents.pkl`) | Danielle Aira Savellano |
| 2. Cleaning and lemmatization (output: `lemmatized_text.xz`) | Anna Ringwood |

AI tool usage is noted at the start of each section.

## 1. File download and text extraction

We pull minutes from the JOB site into `Data/Scraped/`, then extract machine-readable text into `Data/Text/` and assemble `raw_documents.pkl` with meeting identifiers and calendar metadata.

- **Main contributor:** Danielle Aira Savellano
- **AI Acknowledgements:** The following code was written with assistance from GitHub Copilot


## Text preprocessing — PDF, DOC, and DOCX to plain text

Meeting minutes were downloaded from the [JOB website](https://alleghenycontroller.com/the-controller/jail-oversight-board/jail-oversight-board-internal/) as PDF or DOCX files. A CSV lists desired filenames and download links. This chapter:

- Downloads from `Data/Scraped/download_links.csv` into `Data/Scraped` using the naming convention on each row
- Maintains `Data/Scraped/download_tracker.csv` with download status and detected file type

A total of **156 files** were collected (116 PDFs and 40 DOCX), spanning March 2012 through February 2026, including four special meetings.

> Filenames follow `YYYY_M_X_JOB_Minutes` where `YYYY` is year, `M` is month, and `X` indicates a special meeting when `1` (for example `2024_2_1_JOB_Minutes` is February 2024, special).

After download, each file is converted to a UTF-8 `.txt` under `Data/Text/`. **153 of 156** extractions succeeded; three scans produced empty text and were copied to `Data/Error_Files/` for review. Recent minutes sometimes include glossaries and tables that do not convert cleanly; for this project the narrative transcript is the analysis target.


In [1]:
!pip install pypdf python-docx --quiet


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


### I. File Downloading

In [2]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse

import requests

# Paths and Session Setup
BASE = Path("..").resolve()
CSV_IN = BASE / "Data" / "Scraped" / "download_links.csv"
OUT_DIR = BASE / "Data" / "Scraped"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT = OUT_DIR / "download_tracker.csv"
sess = requests.Session()

# Helper Functions
def suffix(url: str) -> str:
    '''Extract file suffix from URL, default to .pdf if not found.'''
    s = Path(urlparse(url).path).suffix.lower()
    return s if s else ".pdf"

def desired_path(title: str, ext: str) -> Path:
    '''Generate desired path given title and extension.'''
    ext = ext if ext.startswith(".") else f".{ext}"
    if ext == ".doc":
        ext = ".docx"  # Convert .doc to .docx
    return OUT_DIR / f"{title}{ext.lower()}"

# Prepare output fields and results list
with CSV_IN.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    cols = list(reader.fieldnames or [])
    rows_in = list(reader)

extra = ["Downloaded", "File Type"]
out_fields = cols + [c for c in extra if c not in cols]
results = []
n_skipped = 0

# Processing input CSV
for row in rows_in:
    title = (row.get("Desired Title")).strip()
    url = (row.get("Link")).strip()
    out = {**row}                                           # Keep original data
    ext = suffix(url)                                       # Get file extension

    target = desired_path(title, ext)                       # Desired path (combines title and extension)

    if target.is_file() and target.stat().st_size > 0:      # If file exists and is not empty
        n_skipped += 1
        out["Downloaded"], out["File Type"] = "True", target.suffix.lower()
        results.append(out)
        continue                                            # Skip download

    try:
        r = sess.get(url, timeout=90)
        r.raise_for_status()
        target.write_bytes(r.content)
        out["Downloaded"], out["File Type"] = "True", ext   # Add download status and file type
    except Exception as e:
        out["Downloaded"], out["File Type"] = "False", ext
        print(f"{title}: {e}")

    results.append(out)                                     # Append result for document
    time.sleep(0.2)

# Write results to output CSV
with CSV_OUT.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=out_fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(results)

# Summary
downloaded = sum(1 for r in results if r.get("Downloaded") == "True")
print(f"{downloaded}/{len(results)} Downloaded ({n_skipped} Already Existing)")
print(f"Saved to: {OUT_DIR}")
print(f"Tracker: {CSV_OUT}")

156/156 Downloaded (156 Already Existing)
Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Scraped
Tracker: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Scraped/download_tracker.csv


In [3]:
# Download Summary

print(f"\n{len(results)} Total Files")

print("\nFile Types")
for file_type in set(r.get("File Type") for r in results):
    count = sum(1 for r in results if r.get("File Type") == file_type)
    print(f"{file_type}: {count}")

print("\nYears")

for year in sorted(set(r.get("Year") for r in results)):
    count = sum(1 for r in results if r.get("Desired Title").split("_")[0] == year)
    print(f"{year}: {count}")


156 Total Files

File Types
.docx: 40
.pdf: 116

Years
2012: 7
2013: 8
2014: 10
2015: 9
2016: 11
2017: 12
2018: 12
2019: 11
2020: 10
2021: 13
2022: 12
2023: 12
2024: 14
2025: 13
2026: 2


In [4]:
special_count = 0

print("\nSpecial Meetings:")
for r in results:
    if r.get("Desired Title").split("_")[2] == "1":
        special_count += 1
        print(r.get("Desired Title"))

print(f"\nTotal Special Meetings: {special_count}")


Special Meetings:
2021_9_1_JOB_Minutes
2024_2_1_JOB_Minutes
2024_3_1_JOB_Minutes
2025_8_1_JOB_Minutes

Total Special Meetings: 4


### II. Text Extraction

In [5]:
from pathlib import Path

# Paths
BASE = Path("..").resolve()
IN_DIR = BASE / "Data" / "Scraped"
OUT_DIR = BASE / "Data" / "Text"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Functions to read, extract, and convert files to plain text.

def read_text_file(path: Path) -> str:
    for encoding in ("utf-8", "utf-8-sig"):
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace")

def extract_pdf_to_text(path: Path) -> str:
    '''Extract text from a PDF file using pypdf. Written by GitHub Copilot.'''
    from pypdf import PdfReader
    # strict=False tolerates slightly malformed PDFs from some scanners/tools.
    reader = PdfReader(str(path), strict=False)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages).strip()

def extract_docx_to_text(path: Path) -> str:
    '''Extract text from a docx file using python-docx. Written by GitHub Copilot.'''
    from docx import Document
    doc = Document(str(path))
    paragraphs = [p.text for p in doc.paragraphs]
    return "\n\n".join(paragraphs).strip()

def file_to_plain_text(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        return extract_pdf_to_text(path)
    if path.suffix.lower() == ".docx":
        return extract_docx_to_text(path)
    return read_text_file(path)

def list_input_files(directory: Path) -> list[Path]:
    # Sorted for reproducible order; skip dotfiles (.DS_Store, etc.).
    return sorted(
        p for p in directory.iterdir()
        if p.is_file() and not p.name.startswith(".")
    )

# Convert each input file to .txt for analysis and skip if output already exists
written: list[Path] = []
n_skipped = 0

for path in list_input_files(IN_DIR):
    # Skip non-document files (e.g., .csv)
    if path.suffix.lower() == ".csv":
        continue
    
    # Skip if the output .txt already exists (avoid re-processing)

    out_path = OUT_DIR / f"{path.stem}.txt"
    
    if out_path.exists() and out_path.stat().st_size == 0:  # Empty file exists
        n_skipped += 1
        written.append(out_path)

        print(f"Warning: {path.stem} already exists but is empty.")
        
        # Make copy of input file in Data/Error_Files/ for manual review if needed.
        error_dir = BASE / "Data" / "Error_Files"
        error_dir.mkdir(parents=True, exist_ok=True)
        error_path = error_dir / path.name
        if not error_path.exists():
            error_path.write_bytes(path.read_bytes())
        continue

    if out_path.exists() and out_path.stat().st_size > 0:
       n_skipped += 1
       written.append(out_path)
       continue                                            # Skip download

    text = file_to_plain_text(path)
    out_path.write_text(text, encoding="utf-8", newline="\n")
    written.append(out_path)

# Summary
print(f"\n{len(written)}/{len(list_input_files(IN_DIR))-2} Files Processed ({n_skipped} Already Existing)")
        
print(f"\nSaved to: {OUT_DIR}")


156/156 Files Processed (156 Already Existing)

Saved to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Text


In [6]:
# Load all converted .txt files for analysis

documents: list[str] = []                               # List of document texts
doc_names: list[str] = []                               # Corresponding list of filenames
for path in sorted(OUT_DIR.glob("*.txt")):
    documents.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(documents)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


### Output 1: raw data in pickle file with date information

In [7]:
# Save all documents in a pickle file for later use in BERT topic modeling and analysis. Include month and year as separate columns for easier analysis.

import pickle
import re

# Creating file path to new "Combined" folder
path = BASE / "Data" / "Combined" / "raw_documents.pkl"
path.parent.mkdir(parents=True, exist_ok=True)

# Extract year and month from filenames (format: YYYY_M_X_JOB_Minutes.txt)
years = []
months = []
for name in doc_names:
    match = re.match(r'(\d{4})_(\d+)_', name)
    if match:
        years.append(int(match.group(1)))
        months.append(int(match.group(2)))
    else:
        years.append(None)
        months.append(None)

# Save documents along with metadata
data = {
    'documents': documents,
    'doc_names': doc_names,
    'years': years,
    'months': months
}

with open(path, 'wb') as f:
    pickle.dump(data, f)

print(f"Saved {len(documents)} documents with metadata to: {path}")

Saved 156 documents with metadata to: /Users/daesavellano/0_CMU/Unstructured Data Analytics/Project/Updated/board_meeting_analysis/Data/Combined/raw_documents.pkl


In [8]:
# Sample loading of raw_documents.pkl
with open(path, 'rb') as f:
    data = pickle.load(f)

# show first row of data
print(data['documents'][0])
print(data['doc_names'][0])
print(data['years'][0])
print(data['months'][0])

The monthly meeting of the Allegheny County Jail Oversight Board was held on Thursday,  
October 4, 2012, in Conference Room #1 of the Court House, Pittsburgh, Pennsylvania at 4:00 p.m. 
 
The following members were present: 
The Honorable Judge Kimberly Clark 
M. Gayle Moss 
Joe Asturi representing The Honorable Chelsa Wagner 
Elliott Howsie, Chief Public Defender 
Austin Davis representing The Honorable Rich Fitzgerald 
Joe Catanese representing The Honorable Dr. Charles Martoni 
The Honorable Sheriff William Mullen 
 
Also present were Warden Orlando Harper, Major Ron Pofi; Marion Damick; Dana Phillips, 
Doug Williams, Chuck Madarino, Larry Ludwig, Mike Gilmore Carol Hertz, ACCHS and other 
interested parties. 
 
 
PUBLIC COMMENT 
 
Mrs. Marion Damick discussed the jail report. She requested the following: 
 Discussed the role of the warden. 
 Would like Warden Harper to concentrate on the library and learning/teaching inmates to 
read. 
 Provided the Warden with a brochure on PA

## 2. Cleaning and lemmatization

Section 1 produced one text file per meeting. Here we load those files, fix encoding and punctuation issues, remove domain-specific stopwords, lemmatize with `spaCy`, and write `lemmatized_text.xz` for LDA and n-gram work.

- **Main contributor:** Anna Ringwood
- **AI Acknowledgements:** The following code was written with assistance from GitHub Copilot


### In this chapter

The following code was written with some assistance from **GitHub Copilot**.

## Text preprocessing — from raw `.txt` to lemmatized documents

This chapter ingests the scraped text files, strips boilerplate and noise, tokenizes and lemmatizes, and exports a single compressed pickle of token lists per document.

### Import libraries and raw document data:


In [9]:
import csv
import time
from pathlib import Path
from urllib.parse import urlparse
import html
import re
import numpy as np

In [10]:
# Set up paths, similar to how we did in the preceding file
BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where the raw text files are kept

In [11]:
raw_docs = []
doc_names = []

# This section's code taken from last block of preceding file with minor modifications to variable names
for path in sorted(OUT_DIR.glob("*.txt")):
    raw_docs.append(path.read_text(encoding="utf-8"))
    doc_names.append(path.name)

print(f"{len(raw_docs)} document(s) → lists `documents` (text) and `doc_names` (filenames)")

156 document(s) → lists `documents` (text) and `doc_names` (filenames)


In [12]:
# Create a helper function to convert document names to their indices when needed (more useful for development than production)
def doc_name_to_idx(name):
    assert name in doc_names, "Document name not found"
    for i, doc_name in enumerate(doc_names):
        if doc_name == name:
            print(f"Document {i} has the title {name}.")

# Test function
doc_name_to_idx("2021_10_0_JOB_Minutes.txt") # returns index 110
doc_names[110]

Document 90 has the title 2021_10_0_JOB_Minutes.txt.


'2022_5_0_JOB_Minutes.txt'

### Remove characters that are ill-handled by `spaCy`:

Before tokenizing with `spaCy`, we'll address some HTML artifacts (e.g., `\xa0`). We'll also replace any instances of white space with a single space. Finally, based on prior exploration, private-use characters (e.g., bullet points from Word documents) get classified as NOUN or PROPN, which makes it difficult to filter them out post-`spaCy`-processing. Therefore, we'll remove those characters from the raw text as well.

In [ ]:
# unescaped_docs = []
# for doc in raw_docs:
#     temp_doc = html.unescape(doc) # unescape HTML entities; courtesy of MS Copilot
#     temp_doc = re.sub(r'\s+', ' ', temp_doc) # also from MS Copilot
#     unescaped_docs.append(temp_doc.strip())

In [13]:
# Check which weird characters are still present
weird_chars = set()
for doc in raw_docs:
    weird_chars.update([char for char in doc if ord(char) > 127]) # "ord(char) > 127" courtesy of Stack Overflow

print(weird_chars)

{'♦', '–', 'ó', '§', '”', 'Ɵ', '•', '\uf0a7', '½', '—', '©', '‘', '\uf0b7', '’', '‐', 'ñ', '…', '“', 'é', 'ć', '\xa0'}


Some of the "weird" characters may very well change the document's meaning if removed (e.g., " ñ ", " ó ", " é "), so we'll keep those and remove all other characters.

In [14]:
# Remove all weird chars except for accented letters from weird_chars
remove_chars = {char for char in weird_chars if char not in {'ñ', 'ó', 'é', 'ć'}}

In [15]:
prelim_docs = []
for doc in raw_docs:
    temp_doc = doc.replace("\n", " ") # replace newlines with spaces
    for char in remove_chars:
        temp_doc = temp_doc.replace(char, " ") # replace each weird character with a space
    temp_doc = " ".join(temp_doc.split()) # replace multiple spaces with a single space - done last so as to not accidentally introduce multiple spaces
    prelim_docs.append(temp_doc)

In [16]:
print(f"Before newline removal: {raw_docs[115][:100]}")
print(f"After newline removal: {prelim_docs[115][:100]}")

Before newline removal: 1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
GALVIN REPORTING SERVICES
412-897-
After newline removal: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-


In [17]:
# Find where the first instance of the bullet character is in the original document to confirm that it was replaced
print(f"Before weird char removal: {raw_docs[1][(raw_docs[1].find("\uf0b7"))-20:(raw_docs[1].find("\uf0b7")+20)]}")
print(f"After weird char removal: {prelim_docs[1][(prelim_docs[1].find("sed the following: ")):(prelim_docs[1].find("Congratulated the")+len("Congratulated the"))]}")

Before weird char removal: sed the following: 
 Congratulated the 
After weird char removal: sed the following: Congratulated the


The newline characters have been replaced with spaces, there are no multi-spaces, and we've removed the unicode characters, so we can use spaCy now to conduct additional preprocessing.

### Remove unneeded transcription text:

In more recent years the minutes are just transcripts with line numbers and transcription company information on each page. We don't need this data, so we filter it out.

In [18]:
# Get indices of the documents that contain "GALVIN REPORTING SERVICES" a.k.a. the ones that contain transcription line numbers + company name we want to remove
transcription_junk = "1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 GALVIN REPORTING SERVICES 412-897-2010 -- 412-461-1838 (FAX)"
docs_w_transcription_junk = []

for i, doc in enumerate(prelim_docs):
    if "GALVIN REPORTING SERVICES" in doc:
        docs_w_transcription_junk.append(i)

In [19]:
# Remove all occurrences of the transcription junk from the documents
for i in docs_w_transcription_junk:
    prelim_docs[i] = prelim_docs[i].replace(transcription_junk, "")

In [20]:
[prelim_docs[i][:50] for i in docs_w_transcription_junk[:10]] # check that junk was removed - scan of the first 10 documents show that the removal was successful

[' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T',
 ' 1 ALLEGHENY COUNTY JAIL OVERSIGHT BOARD MEETING T']

### Remove end matter of minutes

Additionally, the transcribed documents include end matter with vocab glossaries and indices, which would both throw off our analysis and take up unnecessary processing power during lemmatization. We therefore looked for keywords that would indicate the end of the transcription (e.g., "adjournment"). After some trial-and-error, we discovered that all of the transcripted documents except one contained a notary certification at the end, after which the glossary and index appeared. Thus, for these docs we removed all the text after the certification header. For the final document that didn't follow this pattern, we manually trimmed the end matter.

In [21]:
one_occurrence_idxs = []
gt_one_occurrence_idxs = []
lt_one_occurrence_idxs = []
word_phrase = "C E R T I F I C A T E"

for i in docs_w_transcription_junk:
    
    if prelim_docs[i].upper().count(word_phrase) > 1:
        gt_one_occurrence_idxs.append(i)
    elif prelim_docs[i].upper().count(word_phrase) < 1:
        lt_one_occurrence_idxs.append(i)
    else:
        one_occurrence_idxs.append(i)

print(f"One instance ({len(one_occurrence_idxs)}): {one_occurrence_idxs}")
print(f"Less than one instance ({len(lt_one_occurrence_idxs)}): {lt_one_occurrence_idxs}")
print(f"More than one instance ({len(gt_one_occurrence_idxs)}): {gt_one_occurrence_idxs}")

One instance (43): [103, 104, 105, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155]
Less than one instance (1): [119]
More than one instance (0): []


In [22]:
doc_names[119] # upon manual inspection, we'll remove everything after "I, Diane  G. Galvin , a court  reporter"

'2023_2_0_JOB_Minutes.txt'

In [23]:
# Remove all text after "C E R T I F I C A T E" in the documents
for i in one_occurrence_idxs:
    cert_idx = prelim_docs[i].upper().find(word_phrase)
    prelim_docs[i] = prelim_docs[i][:cert_idx] # keep everything up to but not including the word "C E R T I F I C A T E "

# For document 119, remove everything after "I, Diane  G. Galvin"
reporter_idx = prelim_docs[119].upper().find("I, DIANE G. GALVIN")
prelim_docs[119] = prelim_docs[119][:reporter_idx] # keep everything up to but not including the reporter information

In [24]:
# Check that the end matter was successfully removed from all transctiption documents
for i in docs_w_transcription_junk[:10]:
    print(f"Document {i} ends with: {prelim_docs[i][-100:]}") # check the last 100 characters of the document to confirm that the end matter was removed

Document 103 ends with: OWSIE : So moved . The meeting is adjourned . The meeting adjourned at approximately 7:45 p.m.  197 
Document 104 ends with:  HOWSIE : We're adjourned . (Whereupon , the hearing was concluded at approximately 7:22 p.m.)  203 
Document 105 ends with: ry one. Stay safe. See you in the new year. (The meeting concluded at approximately 7:40 p.m.)  224 
Document 115 ends with: rn . JUDGE HOWSIE : Thank you. I second . (Whereupon , the hearing was concluded at 6:49 p.m.)  180 
Document 116 ends with: E : Thank you. The meeting is adjourned . (Whereupon , the meeting was adjourned at 6:49 p.m.)  163 
Document 117 ends with: all.  166 MS. HALLAM : Great job, Terri . (Whereupon , the hearing was adjourned at 6:47 p.m.)  167 
Document 118 ends with: KRAUS : Motion to adjourn . (Whereupon , the meeting was concluded at approximately 7:45 p.m.)  171 
Document 119 ends with: he hearing was concluded at 6:45 p.m.)  157 COMMONWEALTH OF PENNSYLVANIA ) ss COUNTY OF ALLEGHENY ) 


### Final preprocessing with `spaCy`

This section includes removing stop words, tokenizing, and lemmatizing.

After initial exploration in n-grams and LDA, we found additional words and names that should be treated as stop words, which have been added below.

In [25]:
manual_stop_words = [
    # names as found during the n-gram analysis
    'judge court', 'judge joseph', 'judge judge', 'mr .', 'sheriff department',
    'sheriff kevin', 'sheriff office', 'sheriff william', 'warden address',
    'warden administrative', 'warden answer', 'warden chief', 'warden deputy',
    'warden good', 'warden jail', 'warden jason', 'warden judge', 'warden know',
    'warden like', 'warden long', 'warden respond', 'warden staff', 'warden state',
    'warden talk', 'warden think', 'warden update', 'warden want', 'warden warden',
    'warden work', 'warden ms.', 'warden provide', 'warden response', 'warden search',

    # fillers during conversations - should not be meaningful
    'know', 'think', 'want', 'okay', 'thank', 'like',
    'actually', 'thing', 'ask', 'look', 'say', 'yes', 'yeah', 'sure',
    'oh', 'sorry', 'maybe', 'lot', 
    'let', 'bring', 'tell', 'come', 'happen', 'hear',
    'speak', 'try', 'find', 'understand', 'able',
    'way', 'today', 'anybody', 'everybody', 'somebody', 'guy', 'folk',
    'far', 'past', 'long', 'different', 'specific', 'specifically',
    'currently', 'actually', 'forward', 'outside', 'available', 'little',
    'couple', 'ago', 'appreciate', 'mention', 'run', 'add', 'end', 'page',
    
    # some of the single letters are showing up as frequent words
    'e', 't', 's', 'o', 'n', 'r', 'h', 'w', 'c', 'd', 'l', 'g',
    'f', 'p', 'u', 'm', 'y', "'",
    
    # meeting formal words
    'meeting', 'motion', 'second', 'approve', 'adjourn', 'report',
    'member', 'present', 'minute', 'vote', 'board', 'committee',
    'comment', 'public', 'business', 'follow', 'address', 'discussion',
    'record', 'hold', 'begin', 'complete', 'submit', 'review', 'executive', 'president',
    
    # places/locations
    'allegheny', 'county', 'pennsylvania', 'pittsburgh',
    
    # names of specific board members
    'hallam', 'evashavik', 'howsie', 'beasom', 'lazzara', 'brinkman',
    'toma', 'bigley', 'innamorato', 'damick', 'wagner', 'moss',
    'klein', 'perkins', 'williams', 'clark', 'bethany', 'ludwig', 'mcdaniel', 'ron', 'dana', 'chuck', 'doug',
    'larry', 'pofi', 'acchs', 'martoni', 'madarino', 'catanese',
    'phillips', 'donna', 'jo', 'joe', 'orlando', 'harper', 'latoya',
    'mullen', 'fitzgerald', 'asturi', 'carol', 'hertz', 'mike',
    'gilmore', 'marion', 'chelsa', 'william', 'dilucente', '-dilucente', 'cashman',
    'walker', 'corizon', 'wingard', 'trevor', 'defazio', 'korinski',
    'dilucente', 'right', 'talk', 'man', 'hood', 'price', 'griffin', 'officer', 'connor', 'acj', 'mean', 'zak', 'person',
    'visit', 'beth', 'ali', 'kroll', 'davis', 'martin', 'judgeclark', "o'connor", 'kimberly', 'm.', 'gayle', 'elliott',

    # titles, prefixes, and other noticed words/characters with little or no meaning
    '.', '-', 'mr', 'ms', 'mr.', 'ms.', 'dr.', 'dr', 'inmate', 'jail', 'judge', 'warden', 'report', 'oversight', 'board',
    'conference', 'room', 'court', 'honorable', '00', '000', '000incarceratedindividual', '00am', '00o', '00p', '00pm',
    '01', '015', '02', '023', '03', '038', '04', '040', '048', '05', '050', '06', '069',
    'actions', 'mrs', 'following', 'duly', 'unanimously', 'approval', 'agreement', 'share', 'austin', 'party', 'respectfully', 
    'adjournment', 'secretary', 'parole', 'distribute', 'inform', 'mrs.', 'represent'
]

In [26]:
len(manual_stop_words)

274

In [27]:
import spacy
nlp = spacy.load('en_core_web_sm', disable = ['parser', 'ner']) # disable parser and NER so package runs faster

In [28]:
# Define function to keep only the lemmas we care about; this function removes stop words, tokenizes, and lemmatizes all at once
# This function is courtesy of class demos, with minor tweaks
def get_filtered_words(text):
    filtered_lemmas = []
    for token in nlp(text):
        lemma = token.lemma_.lower()
        if not (nlp.vocab[lemma].is_stop
                or lemma in manual_stop_words
                or token.pos_ == 'PUNCT'
                or token.pos_ == 'SPACE'
                or token.pos_ == 'SYM'
                or token.pos_ == 'X') and (
                    # Remove digits, unless that digit is a 4-digit year starting with "19" or "20" (e.g. 1995, 2020)
                    not lemma.isdigit()
                    or (lemma.isdigit() and len(lemma) == 4 and lemma.startswith(('19', '20')))
                ):
            filtered_lemmas.append(lemma)
    return filtered_lemmas

In [29]:
from tqdm import tqdm

documents = []
for prelim_doc in tqdm(prelim_docs): # use tqdm for progress monitoring; time varies depending on your machine
    documents.append(get_filtered_words(prelim_doc))

100%|██████████| 156/156 [03:29<00:00,  1.34s/it]


## Confirm Successful Preprocessing

Finally, we check whether the removal, tokenization, and lemmatization worked correctly by examining the first few documents.

In [30]:
print(documents[:2]) # check the first 2 documents to confirm that the cleaning worked as expected

[['monthly', 'thursday', 'october', '2012', 'house', '4:00', 'p.m.', 'chief', 'defender', 'rich', 'charles', 'sheriff', 'major', 'interested', 'discuss', 'request', 'discuss', 'role', 'concentrate', 'library', 'learning', 'teaching', 'read', 'provide', 'brochure', 'pa', 'prison', 'society', 'supreme', 'rule', 'state', 'pay', 'september', '2012', 'newspaper', 'confinement', 'confine', 'hour', 'solitary', 'confinement', 'solitary', 'confinement', 'finally', 'acting', 'stickman', 'permit', 'voting', 'absentee', 'ballot', 'misdemeanor', 'september', '2012', 'discuss', 'attend', 're-', 'entry', 'state', 'attend', 'sec', 'john', 'wetzel', 'chief', 'probation', 'potinger', 'david', 'hichton', 'u.s.', 'attorney', 'keynote', 'speaker', 'welcome', 'new', 'state', 'promotion', 'warren', 'deputy', 'services', 'program', 'services', 'old', 'naacp', 'floor', 'collect', 'absentee', 'ballot', 'inmates', 'yesterday', 'new', 'correctional', 'health', 'services', 'month', 'collaborative', 'providers', 'f

It appears our preprocessing has been successful. From here, we will save the preprocessed text and then analyze the board meeting minutes via n-grams and topic modeling in subsequent code files.

In [31]:
# Adapted from UDA Preprocessing 20 Newsgroups.ipynb
import pickle

# Creating file path to Text folder
path = OUT_DIR / "lemmatized_text.xz"

# Saving preprocessed text
with open(path, 'wb') as f:
    pickle.dump(documents, f)

In [32]:
# Sample loading of lemmatized_text.xz
with open(path, 'rb') as f:
    documents = pickle.load(f)

print(documents[:2]) # check the first 2 documents to confirm that the cleaning worked as expected


[['monthly', 'thursday', 'october', '2012', 'house', '4:00', 'p.m.', 'chief', 'defender', 'rich', 'charles', 'sheriff', 'major', 'interested', 'discuss', 'request', 'discuss', 'role', 'concentrate', 'library', 'learning', 'teaching', 'read', 'provide', 'brochure', 'pa', 'prison', 'society', 'supreme', 'rule', 'state', 'pay', 'september', '2012', 'newspaper', 'confinement', 'confine', 'hour', 'solitary', 'confinement', 'solitary', 'confinement', 'finally', 'acting', 'stickman', 'permit', 'voting', 'absentee', 'ballot', 'misdemeanor', 'september', '2012', 'discuss', 'attend', 're-', 'entry', 'state', 'attend', 'sec', 'john', 'wetzel', 'chief', 'probation', 'potinger', 'david', 'hichton', 'u.s.', 'attorney', 'keynote', 'speaker', 'welcome', 'new', 'state', 'promotion', 'warren', 'deputy', 'services', 'program', 'services', 'old', 'naacp', 'floor', 'collect', 'absentee', 'ballot', 'inmates', 'yesterday', 'new', 'correctional', 'health', 'services', 'month', 'collaborative', 'providers', 'f